# CNN 图像分类综合项目

## 学习目标

完成一个可离线运行的图像分类项目：数据划分、CNN、训练验证、最佳 checkpoint、测试、推理和错误分析。示例使用合成图像，替换为 MNIST 时只需替换数据集单元。

## 概念模型

训练集用于更新参数，验证集用于选择模型和调度学习率，测试集只在最后评估。所有指标按样本数累计。

In [ ]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from common.engine import train_one_epoch, evaluate
from common.checkpoint import save_checkpoint, load_checkpoint
from common.runtime import seed_everything

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
images = torch.randn(96, 1, 8, 8)
labels = (images[:, :, :4, :4].mean(dim=(1, 2, 3)) > images[:, :, 4:, 4:].mean(dim=(1, 2, 3))).long()
train, val, test = TensorDataset(images[:64], labels[:64]), TensorDataset(images[64:80], labels[64:80]), TensorDataset(images[80:], labels[80:])
train_loader = DataLoader(train, batch_size=16, shuffle=True)
val_loader = DataLoader(val, batch_size=16)
test_loader = DataLoader(test, batch_size=16)
print('splits:', len(train), len(val), len(test))

### 实验 1：模型结构和 shape 契约

**实验目的**：定义小型 CNN 并验证 NCHW 输入经过卷积、池化、AdaptiveAvgPool 和分类头后输出 `(batch,classes)` logits。

逐层推导通道与空间尺寸，确认 Flatten 后特征数与 Linear 输入一致。随机输入只验证结构，不代表分类能力。


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((2, 2)))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(16 * 2 * 2, 2))
    def forward(self, x):
        return self.head(self.features(x))

model = SmallCNN().to(device)
sample = next(iter(train_loader))[0].to(device)
print('input:', sample.shape, 'features:', model.features(sample).shape, 'logits:', model(sample).shape)
print('parameters:', sum(p.numel() for p in model.parameters()))

### 实验 2：训练、最佳 checkpoint 和测试

**实验目的**：完成训练、验证、学习率调度和 best checkpoint 保存。epoch loss 按样本汇总，验证准确率改善时保存模型、optimizer、scheduler 和元数据。

测试前恢复最佳验证权重，而非使用最后一轮。恢复时模型结构和优化组件必须与保存状态兼容。


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.03, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=1)
best = -1.0
history = []
artifact = Path('artifacts/notebook16-cnn.pt')
for epoch in range(3):
    train_result = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_result = evaluate(model, val_loader, loss_fn, device)
    scheduler.step(val_result.accuracy)
    history.append((train_result.loss, val_result.loss, val_result.accuracy))
    if val_result.accuracy > best:
        best = val_result.accuracy
        save_checkpoint(artifact, model, optimizer, epoch=epoch, metrics={'val_accuracy': best}, scheduler=scheduler)
print('history:', history, 'best:', best)

In [ ]:
restored = SmallCNN().to(device)
restored_optimizer = torch.optim.AdamW(restored.parameters(), lr=0.03, weight_decay=1e-4)
restored_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( restored_optimizer, mode='max', patience=1)
metadata = load_checkpoint(artifact, restored, restored_optimizer, map_location=device, scheduler=restored_scheduler)
test_result = evaluate(restored, test_loader, loss_fn, device)
with torch.inference_mode():
    predictions = restored(sample[:3]).argmax(dim=1)
print('restored:', metadata, 'test:', test_result, 'predictions:', predictions.tolist())
assert artifact.exists() and metadata['metrics']['val_accuracy'] == best

### 实验 3：推理、错误分析和对照实验

**实验目的**：打印训练历史、统计样本错误，并明确下一步应进行受控对照实验。错误分析应记录真值、预测、置信度与输入，而不是只报告总体 accuracy。

比较 dropout、增强或 weight decay 时必须固定数据划分、种子和训练预算。


In [ ]:
print('epoch | train_loss | val_loss | val_acc')
for index, row in enumerate(history, 1):
    print(index, *(round(value, 4) for value in row))
errors = (predictions != labels[80:83].to(device)).sum().item()
print('sample errors:', errors, 'experiment: compare dropout=0.0 and dropout=0.1')

## 检查点

解释为什么测试必须加载最佳验证 checkpoint，并列出项目的输入 shape、输出 logits shape、数据集划分和指标。

## 试一试

将 `dropout` 改为 0.0，记录验证准确率；替换为 MNIST 时保持验证/测试预处理不使用训练增强。

## 常见错误与调试

检查 NCHW 维度、标签 dtype、模型和数据 device、最佳 checkpoint 是否真的恢复，以及是否把测试集用于调参。